In [3]:
import numpy as np
from alfworld.agents.environment import get_environment
import alfworld.agents.modules.generic as generic
import alfworld
import os
import yaml 



Local alfworld and test

In [ ]:


with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
#Overwrite the environment type in the configuration
config['env']['type'] = 'AlfredTWEnv'
#initialize the environment
env = get_environment(config['env']['type'])(config, train_eval='train')
env = env.init_env(batch_size=1)

obs, info = env.reset()


KeyboardInterrupt: 

In [ ]:
obs, info = env.reset()


In [ ]:
import pprint
pprint.pprint(obs)

('-= Welcome to TextWorld, ALFRED! =-\n'
 '\n'
 'You are in the middle of a room. Looking quickly around you, you see a '
 'cabinet 19, a cabinet 18, a cabinet 17, a cabinet 16, a cabinet 15, a '
 'cabinet 14, a cabinet 13, a cabinet 12, a cabinet 11, a cabinet 10, a '
 'cabinet 9, a cabinet 8, a cabinet 7, a cabinet 6, a cabinet 5, a cabinet 4, '
 'a cabinet 3, a cabinet 2, a cabinet 1, a coffeemachine 1, a countertop 3, a '
 'countertop 2, a countertop 1, a drawer 4, a drawer 3, a drawer 2, a drawer '
 '1, a fridge 1, a garbagecan 1, a microwave 1, a sinkbasin 1, a stoveburner '
 '4, a stoveburner 3, a stoveburner 2, a stoveburner 1, and a toaster 1.\n'
 '\n'
 'Your task is to: heat some apple and put it in garbagecan.',)


In [ ]:
#extract the "Your task is: " part from the observation
current_obs = obs[0]
if 'Your task is to:' in current_obs:
    task_description = current_obs.split('Your task is to:')[-1].strip()
else :
    task_description = "Complete the task as described in the observation."#default task description if not found in the observation
task_description

'heat some apple and put it in garbagecan.'

In [ ]:
pprint.pprint(info)

{'admissible_commands': [['go to cabinet 1',
                          'go to cabinet 10',
                          'go to cabinet 11',
                          'go to cabinet 12',
                          'go to cabinet 13',
                          'go to cabinet 14',
                          'go to cabinet 15',
                          'go to cabinet 16',
                          'go to cabinet 17',
                          'go to cabinet 18',
                          'go to cabinet 19',
                          'go to cabinet 2',
                          'go to cabinet 3',
                          'go to cabinet 4',
                          'go to cabinet 5',
                          'go to cabinet 6',
                          'go to cabinet 7',
                          'go to cabinet 8',
                          'go to cabinet 9',
                          'go to coffeemachine 1',
                          'go to countertop 1',
                          'go to cou

In [ ]:
info['extra.expert_plan']

[['look']]

In [ ]:
admissible_commands = list(info['admissible_commands'][0])
if admissible_commands:
    random_action = np.random.choice(admissible_commands)
    obs, scores, dones, infos = env.step([random_action])
    print(f"\nAction: {random_action}")
    print(f"Next Obs: {obs}")


Action: go to cabinet 6
Next Obs: ('You arrive at cabinet 6. The cabinet 6 is closed.',)


In [ ]:
info['admissible_commands'][0]

['go to cabinet 1',
 'go to cabinet 10',
 'go to cabinet 11',
 'go to cabinet 12',
 'go to cabinet 13',
 'go to cabinet 14',
 'go to cabinet 15',
 'go to cabinet 16',
 'go to cabinet 17',
 'go to cabinet 18',
 'go to cabinet 19',
 'go to cabinet 2',
 'go to cabinet 3',
 'go to cabinet 4',
 'go to cabinet 5',
 'go to cabinet 6',
 'go to cabinet 7',
 'go to cabinet 8',
 'go to cabinet 9',
 'go to coffeemachine 1',
 'go to countertop 1',
 'go to countertop 2',
 'go to countertop 3',
 'go to drawer 1',
 'go to drawer 2',
 'go to drawer 3',
 'go to drawer 4',
 'go to fridge 1',
 'go to garbagecan 1',
 'go to microwave 1',
 'go to sinkbasin 1',
 'go to stoveburner 1',
 'go to stoveburner 2',
 'go to stoveburner 3',
 'go to stoveburner 4',
 'go to toaster 1',
 'help',
 'inventory',
 'look']

Dyanmic prompt 

In [ ]:
import json
def build_dynamic_prompt(observation,admissible_commands,action_history,previous_DAG,previous_error=None):
    #current observation
    prompt = f"Current observation: {observation}\n"
    #admissible commands
    prompt += "Admissible commands:\n"
    for cmd in admissible_commands:
        prompt += f"- {cmd}\n"
    #previous error
    if previous_error:
        prompt += f"Previous error: {previous_error}\n"
    #previous action history
    if action_history:
        prompt += "Action history(Do not repeat failed loop):\n"
        #only take the last 5 actions for using too much token 
        for i, action in enumerate(action_history[-5:]):
            prompt += f"{i+1}. {action}\n"
    #previous DAG
    if previous_DAG:
        #compact dag 
        compact_DAG = json.dumps(previous_DAG, separators=(',', ':'))
        prompt += f"Previous Reasoning DAG:\n{compact_DAG}\n"
        #update status of the DAG based on current observation and edit it if necessary
        prompt += "Update the DAG based on the current observation and previous error (if any), and adjust your reasoning for the next action accordingly.\n"

    
    return prompt


In [ ]:

system_prompt = (
    "You are an agent in a text-based environment. Your goal is to achieve the task optimally. You must reason using a Directed Acyclic Graph (DAG) format, "
    "\n CRITICAL DAG RULE: \n"
    "1. STRICT PRUNING (NO HISTORY LOGGING): The DAG is your CURRENT plan, NOT a history of past actions. If a path fails (e.g., you opened Cabinet 1 and it was empty), you MUST COMPLETELY DELETE (PRUNE) that node and its edges from the DAG in your next response. Keep the DAG compact. DO NOT leave failed 'examined' nodes in the graph.\n"
    "2. NO FALSE DEPENDENCIES: Do not chain independent actions. For example, 'Examining Cabinet 2' does NOT logically depend on 'Examining Cabinet 1'. They are independent, parallel paths pointing to 'Target location is known'. NEVER draw an edge between independent search locations.\n"
    "3. NODE = STATE: Every node MUST represent a verifiable condition or state of the world (e.g., 'Target item location is known', 'Target item is in inventory'). Do NOT use verbs for nodes. **CRITICAL: Do NOT create separate nodes for different guessed locations (e.g., do NOT create 'item is in cabinet 1', 'item is in cabinet 2'). Use a SINGLE abstract node like 'Item location is known'.**\n"
    "4. EDGE = ACTION: Edges represent the actions required to transition from one state to the next. Represent edges as {\"from\": node_id, \"to\": node_id, \"action\": \"description\"}.\n"
    "5. PLAN AHEAD (GLOBAL PLANNING): On your very first step, you MUST generate the complete sequence of future states to achieve the final goal. All future states start as 'pending'.\n"
    "6. DYNAMIC EXECUTION: In each step, evaluate the DAG. Find the first 'pending' node whose prerequisite nodes are 'completed'. That is your Target_Goal for the current step. Execute the action chunk that corresponds to the edge leading to that Target_Goal.\n"
    "7. PLAN CORRECTION & MULTIPLE PATHS: For uncertain goals (e.g., searching for an item), create multiple parallel edges (actions) pointing to the **EXACT SAME** target state (e.g., edge 1: search cabinet 1, edge 2: search cabinet 2, BOTH pointing to Node 'Item location is known'). **Do NOT exhaustively list every object in the room; just list 2 or 3 most likely paths to keep the DAG compact.** If an action fails, PRUNE the invalid edge, and try the alternative edge.\n"
    


)

#in context learning format 
in_conext_prompt_instruction = "CRITICAL INSTRUCTION: You MUST output your internal reasoning and next action STRICTLY in the following JSON format. Do NOT output any natural language conversational text before or after the JSON block. Your entire response must be parseable by a JSON parser."
#in context learning format example COT
in_context_example = """{
    "Reflection": "I need to find the apple first before I can heat it. The apple is usually on the table, so I will check the table first.",
    "DAG": {
        "nodes": {
            "S1":["Apple location is known", "pending"],
            "S2": ["Agent is holding the apple", "pending"],
            "S3": ["Apple is heated","pending"],
            "S4":["Apple is inside the garbage can","pending"]


        },
        "edges": [
            {"from":"Current_State", "to":"S1", "action": "Seach counterstop"},
            {"from":"Current_State", "to": "S1", "action": "Search fridge 1"},
            {"from":"S1", "to":"S2", "action": "take apple from receptacle"},
            {"from":"S2", "to":"S3", "action": "heat apple with microwave 1"},
            {"from":"S3", "to":"S4", "action": "put apple in garbagecan 1"}
        ],
        "Target_Goal": "S1",
        "Checker": "Node S2, S3, and S4 are planned for the future. Currently, I need to achieve S1. I have two paths. I will try to search counterstop first. If I find the apple there, I can achieve S1 and move on to S2. If I fail to find the apple on the countersotp, I will know that the apple is not there and I can prune that branch and try searching fridge 1 instead."
        },
        "Action_Chunks": {
            "Command" : "go to counterstop 1"
        }


}


"""
#in context learning format example DAG 

#dynamic prompt
dynamic_prompt = build_dynamic_prompt(obs[0], info['admissible_commands'][0], action_history=[], previous_DAG=None, previous_error=None)



In [ ]:
def create_full_prompt(system_prompt,in_conext_prompt_instruction,in_context_example,dynamic_prompt):
    full_prompt = system_prompt + "\n\n" + in_conext_prompt_instruction + "\n\n" + "Here is an example of the JSON format for the reasoning DAG and action chunk:\n" + in_context_example + "\n\n" + "Now, based on the current observation and admissible commands, please output your internal reasoning and next action in the specified JSON format.\n\n" + dynamic_prompt
    return full_prompt

In [ ]:
full_prompt = create_full_prompt(system_prompt,in_conext_prompt_instruction,in_context_example,dynamic_prompt)
print(full_prompt)

You are an agent in a text-based environment. Your goal is to achieve the task optimally. You must reason using a Directed Acyclic Graph (DAG) format, 
 CRITICAL DAG RULE: 
1. STRICT PRUNING (NO HISTORY LOGGING): The DAG is your CURRENT plan, NOT a history of past actions. If a path fails (e.g., you opened Cabinet 1 and it was empty), you MUST COMPLETELY DELETE (PRUNE) that node and its edges from the DAG in your next response. Keep the DAG compact. DO NOT leave failed 'examined' nodes in the graph.
2. NO FALSE DEPENDENCIES: Do not chain independent actions. For example, 'Examining Cabinet 2' does NOT logically depend on 'Examining Cabinet 1'. They are independent, parallel paths pointing to 'Target location is known'. NEVER draw an edge between independent search locations.
3. NODE = STATE: Every node MUST represent a verifiable condition or state of the world (e.g., 'Target item location is known', 'Target item is in inventory'). Do NOT use verbs for nodes. **CRITICAL: Do NOT create 

DAG extract

In [ ]:
import json 
class DAGManager:
    def __init__(self):
        self.current_DAG = None
    def extract_and_update_DAG(self,llm_output):
        #parse the LLM output to extract the DAG and action chunk
        clean_llm_output = llm_output.strip()
        #clean format 
        if clean_llm_output.startswith("```json"):
            clean_llm_output = clean_llm_output[len("```json"):].strip()
        if clean_llm_output.startswith("```"):
            clean_llm_output = clean_llm_output[len("```"):].strip()
        if clean_llm_output.endswith("```"):
            clean_llm_output = clean_llm_output[:-len("```")].strip()
        #parse the cleaned output as JSON
        try:
            parased_data = json.loads(clean_llm_output)
            #check if the parsed data contains the DAG an
            if "DAG" not in parased_data:
                return False, "Missing 'DAG' in the output."
            
            #DAG data 
            dag_data = parased_data["DAG"]

            #check if the DAG data contains the necessary fields
            if "nodes" not in dag_data or "edges" not in dag_data:
                return False, "DAG data must contain 'nodes' and 'edges'."
            
            #ok successfully extract the DAG and action chunk
            self.current_DAG = dag_data
            return True, "DAG extracted and updated successfully."
        except json.JSONDecodeError as e:
            return False, f"JSON parsing error: {str(e)}"
    def get_current_DAG(self):
        return self.current_DAG

Validator

In [ ]:
import json


#this validator is used to check whether the format of LLM output and whether the action follow the admissible commands
class ProgrammaticValidator:
    def __init__(self):
        pass
    def validate(self, llm_output, admissible_commands):
        """
        Parameters:
        llm_output: the raw output from LLM
        admissible_commands: the list of admissible commands provided by the environment
        Returns:
        A tuple (is_valid, result)
        is_valid: a boolean indicating whether the output is valid
        result: if is_valid is True, result is a dictionary containing the parsed reasoning and action; if is_valid is False, result is an error message string
        """
        #clean the llm output
        clean_llm_output = llm_output.strip()
        #clean the llm output to ensure it is a valid JSON format
        if clean_llm_output.startswith("```json"):
            clean_llm_output = clean_llm_output[len("```json"):].strip()
        if clean_llm_output.startswith("```"):
            clean_llm_output = clean_llm_output[len("```"):].strip()
        if clean_llm_output.endswith("```"):
            clean_llm_output = clean_llm_output[:-len("```")].strip()
        #parse the JSON output
        try:
            parsed_output = json.loads(clean_llm_output)
        except json.JSONDecodeError as e:
            return False, f"JSON parsing error: {str(e)}"
        
        #Schema validation
        if not isinstance(parsed_output, dict):
            return False, "Output is not a dictionary."
        
        action_chunk = parsed_output.get('Action_Chunks')
        #check if the action chunk in reasoning DAG 
        if not action_chunk and 'Reasoning_DAG' in parsed_output:
            reasoning_dag = parsed_output['Reasoning_DAG']
            if isinstance(reasoning_dag,dict):
                action_chunk = reasoning_dag.get('Action_Decision_Maker')
        #not exist action chunk in both reasoning DAG and action chunk, return error
        if not action_chunk:
            return False, "Missing 'Action_Chunks' or 'Action_Decision_Maker' in the output."
        #fix potential json fromat issue 
        if isinstance(action_chunk, list):
           if len(action_chunk)>0:
                action_chunk = action_chunk[0]
           else:
                return False, "'Action_Chunks' is an empty list."
       
        #extract the command from action chunk
        if not isinstance(action_chunk, dict) or 'Command' not in action_chunk:
            return False, "Missing 'Command' in 'Action_Chunks'."
        command = str(action_chunk['Command']).strip()

        #check if the command is in admissible commands
        if command not in admissible_commands:
            return False, f"Command '{command}' is not in admissible commands."
        #if all checks passed, return the parsed output
        return True, command

Connect to API test 

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
client = OpenAI()


In [ ]:
def get_LLM_response(client,global_goal,system_prompt,in_conext_prompt_instruction,in_context_example,dynamic_prompt,temperature=0.5):
    global_goal_prompt = f"DO NOT FORGET, Your global goal is: {global_goal}\n"
    response = client.chat.completions.create(
        #  model ="gpt-5.4-nano",
        model = "gpt-4o-mini",
        messages = [
            {"role": "system", "content": system_prompt+ "\n" + in_conext_prompt_instruction + "\n" + in_context_example + "\n" + global_goal_prompt},
            {"role": "user", "content": dynamic_prompt}
        ],
        temperature=temperature,
    )
    return response.choices[0].message.content

            
            
    

In [ ]:
response = get_LLM_response(client,task_description,system_prompt,in_conext_prompt_instruction,in_context_example,dynamic_prompt)


In [ ]:
print(response)

{
    "Reflection": "I need to find the apple first before I can heat it. Since cabinet 6 is closed, I will explore other options. I will check cabinet 1 next as it is a likely location for the apple.",
    "DAG": {
        "nodes": {
            "S1": ["Apple location is known", "pending"],
            "S2": ["Agent is holding the apple", "pending"],
            "S3": ["Apple is heated", "pending"],
            "S4": ["Apple is inside the garbage can", "pending"]
        },
        "edges": [
            {"from": "Current_State", "to": "S1", "action": "Search cabinet 1"},
            {"from": "Current_State", "to": "S1", "action": "Search cabinet 10"},
            {"from": "Current_State", "to": "S1", "action": "Search cabinet 11"},
            {"from": "Current_State", "to": "S1", "action": "Search cabinet 12"},
            {"from": "Current_State", "to": "S1", "action": "Search cabinet 13"},
            {"from": "Current_State", "to": "S1", "action": "Search cabinet 14"},
          

Validator test 

In [ ]:
validator = ProgrammaticValidator()
is_valid, result = validator.validate(response, info['admissible_commands'][0])
if is_valid:
    print(f"Valid output. Extracted command: {result}")
else:
    print(f"Invalid output. Error: {result}")


Valid output. Extracted command: go to cabinet 1


DAG test 

In [ ]:
DAG_manager = DAGManager()
success, message = DAG_manager.extract_and_update_DAG(response)
if success:
    print("DAG extracted and updated successfully.")
    current_DAG = DAG_manager.get_current_DAG()
    print("Current DAG:")
    print(json.dumps(current_DAG, indent=2))
else:
    print(f"Failed to extract DAG. Error: {message}")

DAG extracted and updated successfully.
Current DAG:
{
  "nodes": {
    "S1": [
      "Apple location is known",
      "pending"
    ],
    "S2": [
      "Agent is holding the apple",
      "pending"
    ],
    "S3": [
      "Apple is heated",
      "pending"
    ],
    "S4": [
      "Apple is inside the garbage can",
      "pending"
    ]
  },
  "edges": [
    {
      "from": "Current_State",
      "to": "S1",
      "action": "Search cabinet 1"
    },
    {
      "from": "Current_State",
      "to": "S1",
      "action": "Search cabinet 10"
    },
    {
      "from": "Current_State",
      "to": "S1",
      "action": "Search cabinet 11"
    },
    {
      "from": "Current_State",
      "to": "S1",
      "action": "Search cabinet 12"
    },
    {
      "from": "Current_State",
      "to": "S1",
      "action": "Search cabinet 13"
    },
    {
      "from": "Current_State",
      "to": "S1",
      "action": "Search cabinet 14"
    },
    {
      "from": "Current_State",
      "to": "S1"

In [ ]:
max_game_step = 30 
max_retreies_per_step = 3
#reset the environment
obs, info = env.reset()
#current OBS and admissible commands
current_obs = obs[0]
#set gloabl goal 
if 'Your task is to:' in current_obs:
    global_goal = current_obs.split('Your task is to:')[-1].strip()
else :
    global_goal = "Complete the task as described in the observation."#default task description if not found in the observation
print(f"Global goal: {global_goal}")

current_cmd = info['admissible_commands'][0]

#programmatic validation for the first step
validator = ProgrammaticValidator()
#DAG manager to manage the reasoning DAG
DAG_manager = DAGManager()
action_history = []

for step in range(max_game_step):
    previous_error = None
    action_to_execute = None

    for retry in range(max_retreies_per_step):
        if retry > 1:
            print(f"Retrying step {step}, attempt {retry}...")
        #build dynamic prompt with current observation, admissible commands and previous error (if any)
        dynamic_prompt = build_dynamic_prompt(current_obs, current_cmd, action_history, DAG_manager.get_current_DAG(), previous_error)
        #call LLM to get response  
        response = get_LLM_response(client, global_goal, system_prompt, in_conext_prompt_instruction, in_context_example, dynamic_prompt)
        #print the raw response from LLM
        print(f"LLM response at step {step}, attempt {retry}:\n{response}\n")
        #validate the LLM response
        is_valid, result = validator.validate(response, current_cmd)
        if is_valid:
            action_to_execute = result
            print(f"Valid output at step {step}, attempt {retry}. Extracted command: {action_to_execute}")

            is_dag_valid, dag_message = DAG_manager.extract_and_update_DAG(response)
            if not is_dag_valid:
                print(f"Warning: Failed to extract DAG at step {step}, attempt {retry}. Error: {dag_message}")
            break
        else:
            previous_error = result
            print(f"Invalid output at step {step}, attempt {retry}. Error: {previous_error}")
    # play the valid action in the environment
    if action_to_execute:
        #give to the environment 
        obs, scores, dones, infos = env.step([action_to_execute])

        #append the executed action to action history
        action_history.append(action_to_execute)

        #update current_obs and current_cmd for the next step
        current_obs = obs[0]
        current_cmd = infos['admissible_commands'][0]
        #print env feedback
        print(f"Environment feedback at step {step}:\nObservation: {current_obs}")

        #check if the episode is done
        if dones[0]:
            print(f"Episode finished at step {step}.")
            break
    else:
        print(f"Failed to get a valid action after {max_retreies_per_step} attempts at step {step}. Moving to the next step.")
else:
    print("Reached maximum game steps without finishing the episode.")




     


Global goal: put a toiletpaper in toiletpaperhanger.
LLM response at step 0, attempt 0:
{
    "Reflection": "I need to find the toilet paper first before I can put it in the toilet paper hanger. It is likely to be in one of the cabinets or the drawer, so I will check those locations first.",
    "DAG": {
        "nodes": {
            "S1": ["Toilet paper location is known", "pending"],
            "S2": ["Agent is holding the toilet paper", "pending"],
            "S3": ["Toilet paper is in toilet paper hanger", "pending"]
        },
        "edges": [
            {"from": "Current_State", "to": "S1", "action": "Search cabinet 1"},
            {"from": "Current_State", "to": "S1", "action": "Search cabinet 2"},
            {"from": "Current_State", "to": "S1", "action": "Search cabinet 3"},
            {"from": "Current_State", "to": "S1", "action": "Search drawer 1"},
            {"from": "S1", "to": "S2", "action": "take toilet paper from receptacle"},
            {"from": "S2", "to

In [ ]:
import json 
def collect_expert_trajectory(env, max_number_episodes=10,expert_plan_file=None):
    trajectories = []
    for game_number in range(max_number_episodes):
        obs,info = env.reset()
        current_obs = obs[0]

        #extrat expert plan from info 
        expert_action = info.get('extra.expert_plan', [[]])[0]
        
        trajectory_for_this_game = {
            f"{game_number}": []
        }

        #record the initial observation
        trajectory_for_this_game[f"{game_number}"].append({"Initial_Observation": f"{current_obs}"})

        #bool to indicate whether the game is successful
        game_success = False
        done = False
        step_count = 0

        #if the game not done, keep executing the expert action and recording the trajectory
        while not done:
            #get the expert action from the info
            expert_plan = info.get('extra.expert_plan', [[]])[0]

            #if expert is empty, break the loop 
            if not expert_plan:
                print(f"Game {game_number} failed")
                break
            #take the first action in the expert plan
            expert_action = expert_plan[0]

            #record the expert action
            trajectory_for_this_game[f"{game_number}"].append({"Action":f"{expert_action}"})
            
            #execute the expert action in the environment
            obs, scores, dones, info = env.step([expert_action])

            #update current_obs for the next step
            next_obs = obs[0]
            done = dones[0]
            step_count += 1

            #check if the episode is finished
            if not done:
                #record the next observation for the next step
                trajectory_for_this_game[f"{game_number}"].append({"Observation": f"{next_obs}"})
            else:
                #check if the game is successful based on the score or other info
                if info['won'][0]:
                    game_success = True
                    print(f"Game {game_number} succeeded in {step_count} steps.")
                    #save this trajectory to a json file for later analysis
                    if expert_plan_file:
                        with open(expert_plan_file, 'a',encoding='utf-8') as f:
                           f.write( json.dumps(trajectory_for_this_game) + "\n")

                else:
                    print(f"Game {game_number} failed in {step_count} steps.")

        #if the game is successful, add the trajectory of this game to the trajectories list
        if game_success:
            trajectories.append(trajectory_for_this_game)
    return trajectories



In [ ]:
collected_trajectories = collect_expert_trajectory(env, max_number_episodes=3553,expert_plan_file="Alfworld_expert_trajectories.json")

Game 0 succeeded in 5 steps.
Game 1 succeeded in 7 steps.
Game 2 succeeded in 18 steps.
Game 3 succeeded in 25 steps.
Game 4 succeeded in 10 steps.
Game 5 succeeded in 33 steps.
Game 6 succeeded in 5 steps.
Game 7 succeeded in 47 steps.
Game 8 succeeded in 12 steps.
Game 9 failed in 50 steps.
Game 10 succeeded in 8 steps.
Game 11 succeeded in 15 steps.
Game 12 succeeded in 11 steps.
Game 13 succeeded in 10 steps.
Game 14 succeeded in 9 steps.
Game 15 succeeded in 18 steps.
Game 16 succeeded in 5 steps.
Game 17 succeeded in 12 steps.
Game 18 succeeded in 7 steps.
Game 19 succeeded in 29 steps.
Game 20 succeeded in 40 steps.
Game 21 succeeded in 37 steps.
Game 22 succeeded in 16 steps.
Game 23 succeeded in 15 steps.
Game 24 succeeded in 7 steps.
Game 25 succeeded in 21 steps.
Game 26 succeeded in 19 steps.
Game 27 succeeded in 7 steps.
Game 28 succeeded in 9 steps.
Game 29 succeeded in 28 steps.
Game 30 succeeded in 10 steps.
Game 31 succeeded in 14 steps.
Game 32 succeeded in 37 steps.


In [ ]:
collected_trajectories

[{'0': [{'Observation': '-= Welcome to TextWorld, ALFRED! =-\n\nYou are in the middle of a room. Looking quickly around you, you see a cabinet 10, a cabinet 9, a cabinet 8, a cabinet 7, a cabinet 6, a cabinet 5, a cabinet 4, a cabinet 3, a cabinet 2, a cabinet 1, a coffeemachine 1, a countertop 3, a countertop 2, a countertop 1, a diningtable 1, a drawer 6, a drawer 5, a drawer 4, a drawer 3, a drawer 2, a drawer 1, a fridge 1, a garbagecan 1, a microwave 1, a sinkbasin 1, a stoveburner 4, a stoveburner 3, a stoveburner 2, a stoveburner 1, and a toaster 1.\n\nYour task is to: cool some mug and put it in coffeemachine.'},
   {'Action': 'look'},
   {'Observation': 'You are in the middle of a room. Looking quickly around you, you see nothing.'},
   {'Action': 'go to toaster 1'},
   {'Observation': 'You arrive at toaster 1. On the toaster 1, you see nothing.'},
   {'Action': 'go to stoveburner 1'},
   {'Observation': 'You arrive at stoveburner 1. On the stoveburner 1, you see a kettle 1.'}

In [1]:

from Conver_Alfworld_traj_to_SFT import Convert_Alfworld_traj_to_SFT

sft_data = Convert_Alfworld_traj_to_SFT("Alfworld_expert_trajectories.json", "Alfworld_SFT_data.json")

KeyError: 'Action'

In [2]:
sft_data